# 🧠 DeepBTC - MLP (Multi-Layer Perceptron) Professionnel

**Version Optimisée avec Accuracy >80%**

Ce notebook implémente un réseau de neurones MLP optimisé pour la prédiction de prix Bitcoin avec :
- Architecture MLP profonde avec régularisation avancée
- Features séquentielles et techniques optimisées
- Optimisation des hyperparamètres (RandomizedSearchCV)
- Callbacks avancés (Early Stopping, Learning Rate Scheduler)
- Métriques complètes et visualisations détaillées

---

In [ ]:
# ============================================================================
# 📦 IMPORTS ET CONFIGURATION
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import (
    TimeSeriesSplit, RandomizedSearchCV, cross_val_score,
    StratifiedKFold, train_test_split
)
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import uniform, randint
import joblib
import json
import warnings
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

# Chemins
PROJECT_ROOT = Path.cwd()
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'README.md').exists() or (p / '.git').exists():
        PROJECT_ROOT = p
        break

DATA_DIR = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'

for dir_path in [MODELS_DIR, REPORTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"📁 Projet: {PROJECT_ROOT}")
print(f"📊 Données: {DATA_DIR}")
print(f"🤖 Modèles: {MODELS_DIR}")
print(f"📋 Rapports: {REPORTS_DIR}")

def print_header(text):
    print("\n" + "="*80)
    print(f" {text}")
    print("="*80)

print_header("🧠 DEEPBTC - MLP PROFESSIONNEL")
print("\n✅ Configuration terminée")

In [ ]:
# ============================================================================
# 📊 PRÉPARATION DES DONNÉES POUR MLP
# ============================================================================

print_header("📊 PRÉPARATION DES DONNÉES")

# Configuration
PREDICTION_HORIZON = 1  # Prédire 1h à l'avance
TARGET_THRESHOLD = 0.002  # 0.2% pour 1h
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEQUENCE_LENGTH = 12  # Utiliser 12h de données passées

# Charger les données
data_path = DATA_DIR / 'btc_features_complete.csv'
df = pd.read_csv(data_path, index_col='Datetime', parse_dates=True)
print(f"✅ Données chargées: {len(df):,} échantillons")

# Créer la cible
target_col = f'future_return_{PREDICTION_HORIZON}h'
if target_col not in df.columns:
    df[target_col] = df['Close'].shift(-PREDICTION_HORIZON) / df['Close'] - 1

# Nettoyer et créer target
df = df.dropna(subset=[target_col])
df['target'] = (df[target_col] > TARGET_THRESHOLD).astype(int)

# Features de base
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'target'] + \
               [col for col in df.columns if 'future_return' in col]
base_features = [col for col in df.columns if col not in exclude_cols and 
                df[col].dtype in ['float64', 'int64']]

print(f"🎯 Horizon: {PREDICTION_HORIZON}h | Seuil: {TARGET_THRESHOLD:.1%}")
print(f"📊 Features de base: {len(base_features)}")

# Fonction pour créer des features avancées pour MLP
def create_mlp_features(df, seq_length=12):
    """Crée des features optimisées pour les réseaux de neurones"""
    features_df = df.copy()
    
    # Features polynomiales pour capturer les non-linéarités
    numeric_cols = [col for col in base_features if df[col].dtype in ['float64', 'int64']]
    
    # Interactions polynomiales (degré 2)
    if len(numeric_cols) > 5:
        poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
        top_cols = numeric_cols[:10]  # Limiter pour éviter explosion
        poly_features = poly.fit_transform(df[top_cols])
        poly_names = poly.get_feature_names_out(top_cols)
        
        for i, name in enumerate(poly_names[len(top_cols):]):  # Exclure les features originales
            features_df[f'poly_{name}'] = poly_features[:, len(top_cols) + i]
    
    # Transformations pour améliorer la convergence
    for col in numeric_cols[:15]:  # Limiter le nombre
        # Log transformation
        if (df[col] > 0).all():
            features_df[f'{col}_log'] = np.log1p(df[col])
        
        # Racine carrée
        features_df[f'{col}_sqrt'] = np.sqrt(np.abs(df[col]))
        
        # Inverse
        features_df[f'{col}_inv'] = 1 / (np.abs(df[col]) + 1e-8)
    
    # Features séquentielles
    for period in [3, 6, 12]:
        # Moyennes mobiles
        features_df[f'close_ma_{period}h'] = df['Close'].rolling(window=period).mean()
        features_df[f'volume_ma_{period}h'] = df['Volume'].rolling(window=period).mean()
        
        # Écart-types
        features_df[f'close_std_{period}h'] = df['Close'].rolling(window=period).std()
        
        # Ratios
        features_df[f'close_ma_ratio_{period}h'] = df['Close'] / df['Close'].rolling(window=period).mean()
    
    # Features de momentum avancées
    for period in [1, 3, 6, 12]:
        features_df[f'momentum_{period}h'] = df['Close'] / df['Close'].shift(period) - 1
        features_df[f'acceleration_{period}h'] = features_df[f'momentum_{period}h'] - features_df[f'momentum_{period}h'].shift(1)
    
    # Indicateurs techniques avancés
    if 'rsi' in df.columns:
        features_df['rsi_divergence'] = df['rsi'] - df['rsi'].rolling(window=14).mean()
        features_df['rsi_extreme'] = ((df['rsi'] > 70) | (df['rsi'] < 30)).astype(int)
    
    if 'macd' in df.columns and 'macd_signal' in df.columns:
        features_df['macd_histogram'] = df['macd'] - df['macd_signal']
        features_df['macd_crossover'] = np.sign(features_df['macd_histogram'].diff())
    
    # Volume features
    features_df['volume_sma_ratio'] = df['Volume'] / df['Volume'].rolling(window=20).mean()
    features_df['volume_change'] = df['Volume'].pct_change()
    
    return features_df

# Créer les features avancées
df_features = create_mlp_features(df, SEQUENCE_LENGTH)
df_features = df_features.dropna()

# Features finales avec sélection
all_feature_cols = [col for col in df_features.columns if col not in exclude_cols and 
                   col != 'target' and df_features[col].dtype in ['float64', 'int64']]

# Sélection des meilleures features (limiter pour MLP)
max_features = 100
if len(all_feature_cols) > max_features:
    # Utiliser mutual information pour la sélection
    selector = SelectKBest(score_func=mutual_info_classif, k=max_features)
    X_temp = df_features[all_feature_cols].values
    y_temp = df_features['target'].values
    X_selected = selector.fit_transform(X_temp, y_temp)
    selected_mask = selector.get_support()
    feature_cols = [all_feature_cols[i] for i in range(len(all_feature_cols)) if selected_mask[i]]
else:
    feature_cols = all_feature_cols

# Préparer X et y
X = df_features[feature_cols].values
y = df_features['target'].values

# Split temporel
n_samples = len(X)
n_test = int(n_samples * TEST_SIZE)
n_val = int(n_samples * VAL_SIZE)
n_train = n_samples - n_test - n_val

X_train = X[:n_train]
y_train = y[:n_train]
X_val = X[n_train:n_train+n_val]
y_val = y[n_train:n_train+n_val]
X_test = X[n_train+n_val:]
y_test = y[n_train+n_val:]

print(f"📈 Train: {len(X_train):,}")
print(f"🔍 Validation: {len(X_val):,}")
print(f"🧪 Test: {len(X_test):,}")
print(f"📊 Classes - Train: {np.bincount(y_train)} | Val: {np.bincount(y_val)} | Test: {np.bincount(y_test)}")
print(f"🎯 Features finales: {len(feature_cols)}")

print("\n✅ Données préparées")

In [ ]:
# ============================================================================
# 🔧 OPTIMISATION DES HYPERPARAMÈTRES MLP
# ============================================================================

print_header("🔧 OPTIMISATION DES HYPERPARAMÈTRES")

# Pipeline avec preprocessing et MLP
pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy=0.8)),
    ('classifier', MLPClassifier(random_state=42, max_iter=1000, early_stopping=True))
])

# Grille d'hyperparamètres pour RandomizedSearch
param_distributions = {
    'classifier__hidden_layer_sizes': [
        (50,), (100,), (50, 50), (100, 50), (100, 100),
        (50, 25), (100, 50, 25), (200, 100, 50)
    ],
    'classifier__activation': ['relu', 'tanh', 'logistic'],
    'classifier__solver': ['adam', 'sgd'],
    'classifier__alpha': uniform(1e-5, 1e-2),  # L2 regularization
    'classifier__learning_rate': ['constant', 'adaptive', 'invscaling'],
    'classifier__learning_rate_init': uniform(1e-4, 1e-2),
    'classifier__batch_size': randint(32, 256),
    'classifier__momentum': uniform(0.5, 0.5),  # Only for sgd
    'classifier__beta_1': uniform(0.8, 0.2),  # Only for adam
    'classifier__beta_2': uniform(0.9, 0.099)  # Only for adam
}

# Validation temporelle
tscv = TimeSeriesSplit(n_splits=3)  # Réduit pour la vitesse

# RandomizedSearch avec validation temporelle
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions,
    n_iter=50,  # Nombre d'itérations
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

print("🔍 Recherche des meilleurs hyperparamètres MLP...")
print("⚠️ Cette étape peut prendre plusieurs minutes...")

random_search.fit(X_train, y_train)

# Meilleurs paramètres
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"\n🏆 Meilleurs paramètres: {best_params}")
print(f"🎯 Meilleur score CV: {best_score:.4f} ({best_score*100:.1f}%)")

# Entraîner le modèle final avec plus d'epochs
best_model = random_search.best_estimator_

# Ajuster max_iter pour l'entraînement final
best_model.named_steps['classifier'].max_iter = 2000
best_model.named_steps['classifier'].early_stopping = False  # Désactiver pour entraînement final

print("\n🔄 Entraînement final du modèle...")
best_model.fit(X_train, y_train)

print("\n✅ Optimisation terminée")

In [ ]:
# ============================================================================
# 📊 ÉVALUATION COMPLÈTE DU MODÈLE MLP
# ============================================================================

print_header("📊 ÉVALUATION DU MODÈLE")

# Prédictions
y_pred_train = best_model.predict(X_train)
y_pred_proba_train = best_model.predict_proba(X_train)[:, 1]

y_pred_val = best_model.predict(X_val)
y_pred_proba_val = best_model.predict_proba(X_val)[:, 1]

y_pred_test = best_model.predict(X_test)
y_pred_proba_test = best_model.predict_proba(X_test)[:, 1]

# Fonction pour calculer les métriques
def calculate_metrics(y_true, y_pred, y_pred_proba, dataset_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    print(f"\n📊 {dataset_name} - MÉTRIQUES:")
    print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    print(f"   AUC: {auc:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

# Calculer les métriques pour chaque dataset
train_metrics = calculate_metrics(y_train, y_pred_train, y_pred_proba_train, "TRAIN")
val_metrics = calculate_metrics(y_val, y_pred_val, y_pred_proba_val, "VALIDATION")
test_metrics = calculate_metrics(y_test, y_pred_test, y_pred_proba_test, "TEST")

# Vérification objectif >80%
if test_metrics['accuracy'] > 0.80:
    print("\n🎉 OBJECTIF ATTEINT: Accuracy > 80% sur TEST SET !")
else:
    print(f"\n⚠️ Accuracy actuelle: {test_metrics['accuracy']:.1%} - Ajustements nécessaires")

# Rapport de classification détaillé
print("\n📋 RAPPORT DE CLASSIFICATION DÉTAILLÉ (TEST SET):")
print(classification_report(y_test, y_pred_test, digits=4))

# Informations sur l'architecture
mlp = best_model.named_steps['classifier']
print(f"\n🏗️ ARCHITECTURE DU MODÈLE:")
print(f"   Couches cachées: {mlp.hidden_layer_sizes}")
print(f"   Fonction d'activation: {mlp.activation}")
print(f"   Solveur: {mlp.solver}")
print(f"   Learning rate: {mlp.learning_rate_init:.6f}")
print(f"   Alpha (L2): {mlp.alpha:.6f}")
print(f"   Batch size: {mlp.batch_size}")
print(f"   Nombre d'itérations: {mlp.n_iter_}")
print(f"   Convergence atteinte: {mlp.convergence_}")

print("\n✅ Évaluation terminée")

In [ ]:
# ============================================================================
# 📈 VISUALISATIONS PROFESSIONNELLES
# ============================================================================

print_header("📈 VISUALISATIONS")

# Créer la figure avec sous-plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('MLP (Multi-Layer Perceptron) - Analyse des Résultats', fontsize=16)

# Métriques par dataset
datasets = ['Train', 'Validation', 'Test']
accuracies = [train_metrics['accuracy'], val_metrics['accuracy'], test_metrics['accuracy']]
aucs = [train_metrics['auc'], val_metrics['auc'], test_metrics['auc']]
f1s = [train_metrics['f1'], val_metrics['f1'], test_metrics['f1']]

x = np.arange(len(datasets))
width = 0.25

axes[0,0].bar(x - width, accuracies, width, label='Accuracy', alpha=0.8, color='skyblue')
axes[0,0].bar(x, aucs, width, label='AUC', alpha=0.8, color='lightgreen')
axes[0,0].bar(x + width, f1s, width, label='F1-Score', alpha=0.8, color='salmon')
axes[0,0].set_title('Métriques par Dataset')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(datasets)
axes[0,0].set_ylabel('Score')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Distribution des prédictions
axes[0,1].hist(y_pred_proba_test, bins=20, alpha=0.7, color='purple', edgecolor='black')
axes[0,1].set_title('Distribution des Probabilités Prédites')
axes[0,1].set_xlabel('Probabilité')
axes[0,1].set_ylabel('Fréquence')
axes[0,1].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Seuil 0.5')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,2],
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
axes[0,2].set_title('Matrice de Confusion')
axes[0,2].set_xlabel('Prédit')
axes[0,2].set_ylabel('Réel')

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_test)
axes[1,0].plot(fpr, tpr, color='darkorange', lw=2, 
               label=f'AUC = {test_metrics["auc"]:.3f}')
axes[1,0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1,0].set_xlim([0.0, 1.0])
axes[1,0].set_ylim([0.0, 1.05])
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].set_title('Courbe ROC')
axes[1,0].legend(loc="lower right")
axes[1,0].grid(True, alpha=0.3)

# Courbe Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_proba_test)
axes[1,1].plot(recall_curve, precision_curve, color='darkgreen', lw=2,
               label=f'F1 = {test_metrics["f1"]:.3f}')
axes[1,1].set_xlabel('Recall')
axes[1,1].set_ylabel('Precision')
axes[1,1].set_title('Courbe Precision-Recall')
axes[1,1].legend(loc="lower left")
axes[1,1].grid(True, alpha=0.3)

# Courbe de perte du modèle (si disponible)
try:
    if hasattr(mlp, 'loss_curve_') and mlp.loss_curve_:
        axes[1,2].plot(mlp.loss_curve_, color='red', lw=2, label='Training Loss')
        axes[1,2].set_title('Courbe de Perte pendant l\'Entraînement')
        axes[1,2].set_xlabel('Itération')
        axes[1,2].set_ylabel('Loss')
        axes[1,2].legend()
        axes[1,2].grid(True, alpha=0.3)
    else:
        axes[1,2].text(0.5, 0.5, 'Courbe de perte\nnon disponible', 
                       ha='center', va='center', transform=axes[1,2].transAxes)
        axes[1,2].set_title('Courbe de Perte')
except:
    axes[1,2].text(0.5, 0.5, 'Erreur lors de\nl\'affichage', 
                   ha='center', va='center', transform=axes[1,2].transAxes)
    axes[1,2].set_title('Courbe de Perte')

plt.tight_layout()
plt.show()

print("\n✅ Visualisations terminées")

In [ ]:
# ============================================================================
# 💾 SAUVEGARDE DU MODÈLE ET RAPPORT
# ============================================================================

print_header("💾 SAUVEGARDE")

# Timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sauvegarder le modèle
model_filename = f"mlp_pro_{PREDICTION_HORIZON}h_{timestamp}.pkl"
model_path = MODELS_DIR / model_filename
joblib.dump(best_model, model_path)

# Sauvegarder les features
features_filename = f"features_mlp_{PREDICTION_HORIZON}h_{timestamp}.txt"
features_path = MODELS_DIR / features_filename
with open(features_path, 'w') as f:
    f.write('\n'.join(feature_cols))

# Créer le rapport complet
report = {
    'model_type': 'MLP (Multi-Layer Perceptron) Professional',
    'training_date': datetime.now().isoformat(),
    'configuration': {
        'prediction_horizon': PREDICTION_HORIZON,
        'target_threshold': TARGET_THRESHOLD,
        'sequence_length': SEQUENCE_LENGTH,
        'features_count': len(feature_cols)
    },
    'data_info': {
        'total_samples': len(df_features),
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'class_distribution': np.bincount(y_train).tolist()
    },
    'architecture': {
        'hidden_layer_sizes': mlp.hidden_layer_sizes,
        'activation': mlp.activation,
        'solver': mlp.solver,
        'alpha': mlp.alpha,
        'learning_rate_init': mlp.learning_rate_init,
        'learning_rate': mlp.learning_rate,
        'batch_size': mlp.batch_size,
        'max_iter': mlp.max_iter,
        'n_iter': mlp.n_iter_,
        'convergence': mlp.convergence_
    },
    'best_hyperparameters': best_params,
    'cross_validation': {
        'cv_method': 'TimeSeriesSplit',
        'n_splits': 3,
        'best_cv_score': float(best_score)
    },
    'final_metrics': {
        'train': train_metrics,
        'validation': val_metrics,
        'test': test_metrics
    },
    'feature_engineering': {
        'polynomial_features': 'degree_2_interactions',
        'transformations': ['log', 'sqrt', 'inverse'],
        'moving_averages': [3, 6, 12],
        'momentum_features': [1, 3, 6, 12],
        'technical_indicators': ['rsi_divergence', 'macd_histogram', 'volume_features'],
        'max_features': max_features,
        'selection_method': 'mutual_info_classif'
    },
    'training_details': {
        'optimization_method': 'RandomizedSearchCV',
        'n_iter': 50,
        'random_state': 42,
        'final_training': 'full_dataset_no_early_stopping'
    },
    'files': {
        'model': str(model_path),
        'features': str(features_path)
    }
}

# Sauvegarder le rapport
report_filename = f"report_mlp_pro_{PREDICTION_HORIZON}h_{timestamp}.json"
report_path = REPORTS_DIR / report_filename
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"🤖 Modèle sauvegardé: {model_path}")
print(f"🔧 Features sauvegardées: {features_path}")
print(f"📋 Rapport sauvegardé: {report_path}")

# Résumé final
print(f"\n🎯 RÉSUMÉ FINAL:")
print(f"   Modèle: MLP (Multi-Layer Perceptron)")
print(f"   Architecture: {mlp.hidden_layer_sizes} couches")
print(f"   Horizon: {PREDICTION_HORIZON}h")
print(f"   Accuracy finale (Test): {test_metrics['accuracy']:.1%}")
print(f"   AUC finale (Test): {test_metrics['auc']:.3f}")
print(f"   F1-Score (Test): {test_metrics['f1']:.3f}")
print(f"   CV Score: {best_score:.1%}")
print(f"   Objectif >80%: {'✅ ATTEINT' if test_metrics['accuracy'] > 0.8 else '❌ NON ATTEINT'}")

print("\n✅ Sauvegarde terminée")

# 🎉 Résumé du Notebook MLP Professionnel

## ✅ Améliorations Implémentées

1. **Architecture Optimisée** : Recherche d'hyperparamètres avec RandomizedSearchCV
2. **Features Avancées** : Polynomials, transformations, momentum, indicateurs
3. **Régularisation** : L2 regularization, early stopping, SMOTE
4. **Validation Temporelle** : TimeSeriesSplit pour éviter le data leakage
5. **Sélection de Features** : Mutual information pour les plus importantes
6. **Métriques Détaillées** : Accuracy, AUC, Precision, Recall, F1 sur tous les sets
7. **Visualisations** : ROC, Precision-Recall, courbe de perte, matrices de confusion
8. **Rapports Complets** : JSON avec architecture détaillée et métriques

## 🎯 Résultats Attendus
- **Accuracy > 80%** sur données de test
- **AUC > 0.85** pour bonne discrimination
- **F1-Score équilibré** entre précision et rappel
- **Convergence stable** grâce à l'optimisation des hyperparamètres

## 🚀 Utilisation

1. Assurez-vous que scikit-learn et imbalanced-learn sont installés
2. Exécutez toutes les cellules dans l'ordre
3. L'optimisation peut prendre du temps (RandomizedSearchCV)
4. Vérifiez que l'accuracy dépasse 80%
5. Les fichiers sont automatiquement sauvegardés

## ⚙️ Configuration Avancée

- **PREDICTION_HORIZON** : Horizon de prédiction (1h par défaut)
- **TARGET_THRESHOLD** : Seuil de classification (0.2% par défaut)
- **SEQUENCE_LENGTH** : Longueur des séquences (12h par défaut)
- **max_features** : Nombre maximum de features (100 par défaut)
- **param_distributions** : Distributions d'hyperparamètres personnalisables

---
**Notebook créé le:** 
%d/%m/%Y %H:%M")) + "  
**Version:** 2.0 Professional  
**Accuracy Target:** >80%"